In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from tqdm import tqdm

from src.utils.data import LocalUserScaler, get_windows

In [2]:
df = get_windows(window_size=7)

In [3]:
agg_rules = {
    "mood": ["count", "mean", "std"],
    "circumplex.arousal": ["count", "mean", "std"],
    "circumplex.valence": ["count", "mean", "std"],
    "activity": ["count", "mean", "std"],
    "call": ["count"],
    "sms": ["count"],
    "screen": ["count", "sum", "mean", "std"],
    "appCat.builtin": ["count", "sum", "mean", "std"],
    "appCat.communication": ["count", "sum", "mean", "std"],
    "appCat.entertainment": ["count", "sum", "mean", "std"],
    "appCat.finance": ["count", "sum", "mean", "std"],
    "appCat.game": ["count", "sum", "mean", "std"],
    "appCat.office": ["count", "sum", "mean", "std"],
    "appCat.other": ["count", "sum", "mean", "std"],
    "appCat.social": ["count", "sum", "mean", "std"],
    "appCat.travel": ["count", "sum", "mean", "std"],
    "appCat.unknown": ["count", "sum", "mean", "std"],
    "appCat.utilities": ["count", "sum", "mean", "std"],
    "appCat.weather": ["count", "sum", "mean", "std"],
}

In [4]:
def get_window_skeleton(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates a skeleton DataFrame that fills each day in the window.
        for each date in the window based on the `window_size_days` column in the input dataframe.
        (note: it ensures that all dates would be present even if there are no readings in the source dataframe)

    This dataframe should be used to join feature aggregates to ensure that the windows will be
    of the same sequence length.

    Args:
        df (pd.DataFrame): The input dataframe containing window information.

    Returns:
        pd.DataFrame: A skeleton dataframe with the following columns:
            - window_id: Unique identifier for each window.
            - id: ID of the patient.
            - split: The data split (e.g., train, test, val).
            - date: Each date in the window based on the `window_size_days` column in the input dataframe
            - target_mean_mood: The mean mood for the target date of the window.
    """
    window_size = df["window_size_days"].iloc[0]
    unique_windows = df.drop_duplicates(subset=["window_id"])[
        ["window_id", "id", "split", "date_target", "target_mean_mood"]
    ]

    offsets = pd.DataFrame({"day_offset": range(1, window_size + 1)})

    skeleton = unique_windows.merge(offsets, how="cross")

    skeleton["date"] = pd.to_datetime(skeleton["date_target"]) - pd.to_timedelta(
        skeleton["day_offset"], unit="D"
    )

    return skeleton[["window_id", "id", "split", "date", "target_mean_mood"]]

In [5]:
def compute_daily_aggregates(df: pd.DataFrame, agg_rules: dict) -> pd.DataFrame:
    """Compute daily aggregates for the given DataFrame.
    Will be used to compute the features for the RNN model.
    Each row will contain a daily aggregates for all variable types (mood, activity, calls, etc.) for a specific window and date.

    Args:
        df (pd.DataFrame): dataframe containing windowed events data (tall format).
            E.g. output from the `src.utils.data.get_windows` function

        agg_rules (dict): aggregation rules specifying which statistics to compute for each variable.
            E.g. {
                "mood": ["count", "meanstd"],
                "circumplex.arousal": ["count", "mean", "std"],
                ...

    Returns:
        pd.DataFrame: dataframe containing:
        - daily aggregates for each variable and window.
        - cyclically encoded date features (day_sin, day_cos).
        - mean mood for the target date of the window (target_mean_mood).
    """

    df = df.copy()

    # compute daily aggregates
    daily_agg = df.groupby(["window_id", "date", "variable"])["value"].agg(
        ["count", "mean", "sum", "std"]
    )
    wide_df = daily_agg.unstack("variable")

    # filter out only the aggregates we need
    valid_columns = []
    for stat, var in wide_df.columns:
        if var in agg_rules and stat in agg_rules[var]:
            valid_columns.append((stat, var))

    wide_df = wide_df[valid_columns]

    # Flatten names (e.g., 'mood_mean', 'sms_sum')
    wide_df.columns = [f"{var}_{stat}" for stat, var in wide_df.columns]
    wide_df = wide_df.reset_index()

    # ensure the windows have the same sequence lengths)
    window_skeleton = get_window_skeleton(df)
    complete_wide_df = window_skeleton.merge(
        wide_df, on=["window_id", "date"], how="left"
    )

    # Add the missingness indicators
    for col in agg_rules.keys():
        count_key = f"{col}_count"
        complete_wide_df[f"{col}_mask"] = complete_wide_df[count_key].isna().astype(int)

    # impute missing values
    complete_wide_df = complete_wide_df.fillna(0)

    # add the cyclical date features
    day_of_week = complete_wide_df["date"].dt.dayofweek
    complete_wide_df["day_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    complete_wide_df["day_cos"] = np.cos(2 * np.pi * day_of_week / 7)

    return complete_wide_df

In [6]:
feature_df = compute_daily_aggregates(df, agg_rules)

train_df = feature_df[feature_df["split"] == "train"]
val_df = feature_df[feature_df["split"] == "val"]
test_df = feature_df[feature_df["split"] == "test"]

In [7]:
features_to_scale = [f"{k}_{agg_func}" for k, v in agg_rules.items() for agg_func in v]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", LocalUserScaler(id_col="id"), ["id"] + features_to_scale),
        ("label", LocalUserScaler(id_col="id"), ["id", "target_mean_mood"]),
        ("user_id", "passthrough", ["id"]),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

pipeline = Pipeline(steps=[("scaler", preprocessor)])
pipeline.set_output(transform="pandas")
pipeline.fit(train_df)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('label', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [8]:
def decode_label(preprocessor: Pipeline, user_id: str, encoded_label: float) -> float:
    """Decode the encoded label back to its original scale.

    Args:
        preprocessor (Pipeline): The fitted preprocessing pipeline.
        user_id (int): The oridinally encoded user index.
        encoded_label (float): The encoded label value to decode.

    Returns:
        float: The decoded label value in the original scale.
    """

    mean, std = preprocessor.named_transformers_["label"].stats_[user_id][
        "target_mean_mood"
    ]

    return encoded_label * std + mean

In [9]:
train_df = pipeline.transform(train_df)
val_df = pipeline.transform(val_df)
test_df = pipeline.transform(test_df)

In [10]:
import torch
from torch.utils.data import DataLoader, Dataset


class RnnDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
    ):
        self.X = []
        self.y = []
        self.user_ids = []
        for name, group in df.groupby("window_id"):
            self.X.append(
                group.drop(
                    columns=["id", "window_id", "date", "split", "target_mean_mood"]
                ).values
            )
            self.y.append(group["target_mean_mood"].iloc[0])
            self.user_ids.append(group["id"].iloc[0])

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.float32),
            self.user_ids[idx],
        )

In [11]:
train_dataset = RnnDataset(train_df)
val_dataset = RnnDataset(val_df)

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=1,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=1,
)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class RnnModel(nn.Module):
    def __init__(self, n_users, n_features, hidden_dim=64):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            batch_first=True,
            num_layers=3,
            dropout=0.2,
        )

        # --- ATTENTION LAYER ---
        # This layer looks at each hidden state and assigns it a "score"
        self.attention_weights = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(0.1)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.SELU(),
            nn.AlphaDropout(0.3),
            nn.Linear(128, 64),
            nn.SELU(),
            nn.Linear(64, 1),
        )

        self._init_weights()

    def _init_weights(self):
        """Initializes weights for the SELU-based regressor using LeCun Normal."""
        for m in self.regressor.modules():
            if isinstance(m, nn.Linear):
                # Using PyTorch's built-in trick for LeCun Normal
                nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="linear")

                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, seq_data):
        lstm_out, _ = self.lstm(seq_data)
        energy = self.attention_weights(lstm_out)
        weights = F.softmax(energy, dim=1)
        context_vector = torch.sum(lstm_out * weights, dim=1)

        combined = self.dropout(context_vector)
        return self.regressor(combined)

In [ ]:
def baseline(X, preprocessor):
    y_true = []
    y_pred = []
    user_ids = []

    for name, group in X.groupby("window_id"):
        y_true.append(group["target_mean_mood"].iloc[0])
        user_ids.append(group["id"].iloc[0])
        y_pred.append(group.iloc[-1]["mood_mean"])

    decoded_labels = [
        decode_label(preprocessor, user_idx, label)
        for label, user_idx in zip(y_true, user_ids)
    ]
    decoded_predictions = [
        decode_label(preprocessor, user_idx, prediction)
        for prediction, user_idx in zip(y_pred, user_ids)
    ]

    print(
        f"RMSE: {np.mean((np.array(decoded_predictions) - np.array(decoded_labels)) ** 2) ** 0.5:.2f} Avg Mood Points"
    )

In [45]:
baseline(train_df, preprocessor)
baseline(val_df, preprocessor)
baseline(test_df, preprocessor)

RMSE: 0.99 Avg Mood Points
RMSE: 1.16 Avg Mood Points
RMSE: 0.97 Avg Mood Points


In [46]:
class EarlyStopping:
    def __init__(self, path: Path, patience: int = 3):
        """Early stopping helper class to save the best model during training.

        Args:
            patience (int, optional): Number of epochs with no improvement
                after which training will be stopped. Defaults to 3.
            path (str, optional): Path to save the best model. Defaults to "best_model.pth".
        """
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_loss = float("inf")

    def __call__(self, val_loss: float, model: torch.nn.Module) -> bool:
        """Check if early stopping condition is met.

        Args:
            val_loss (float): current validation loss
            model (torch.nn.Module): model to save

        Returns:
            bool: True if training should be stopped, False otherwise.
        """
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0

            torch.save(model, self.path)
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
            return False

In [ ]:
model = RnnModel(
    n_features=len(
        train_df.columns.drop(["id", "window_id", "date", "target_mean_mood", "split"])
    ),
    n_users=train_df["id"].nunique(),
    hidden_dim=64,
)

In [49]:
import torch.optim as optim

save_path = Path("./best_rnn_model.pt")

In [50]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()
stopper = EarlyStopping(patience=3, path=save_path)

In [ ]:
epochs = 100
for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0

    pbar = tqdm(train_loader, desc=f"Training Epoch {epoch + 1}/{epochs}")
    for x, y, user_idx in pbar:
        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y.view(-1, 1))

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)

    # --- VALIDATION ---
    model.eval()
    val_mse = 0
    val_unscaled_mse = 0
    val_total_samples = 0

    pbar = tqdm(val_loader, desc=f"Validation Epoch: {epoch + 1}/{epochs}")
    with torch.no_grad():
        for x, y, u in pbar:
            # outputs = model(x, u)
            outputs = model(x)
            val_mse += criterion(outputs, y.view(-1, 1)).item()

            # calculate unscaled MSE for interpretability
            outputs = outputs.squeeze().tolist()
            labels = y.squeeze().tolist()
            user_indexes = u

            unscaled_labels = [
                decode_label(preprocessor, user_idx, label)
                for label, user_idx in zip(labels, user_indexes)
            ]

            unscaled_predictions = [
                decode_label(preprocessor, user_idx, output)
                for output, user_idx in zip(outputs, user_indexes)
            ]

            val_unscaled_mse += np.sum(
                (np.array(unscaled_predictions) - np.array(unscaled_labels)) ** 2
            )
            val_total_samples += len(labels)

    avg_val_loss = val_mse / len(val_loader)
    avg_val_rmse = (val_unscaled_mse / val_total_samples) ** 0.5

    print(f">> Epoch {epoch + 1} Results:")
    print(f"   Train MSE: {avg_train_loss:.2f}")
    print(
        f"   Val   MSE: {avg_val_loss:.2f} | RMSE: {avg_val_rmse:.2f} Avg Mood Points"
    )
    print("\n")

    # --- EARLY STOPPING CHECK ---
    if stopper(avg_val_loss, model):
        print(
            f"\nEarly stopping triggered. No improvement for {stopper.patience} epochs."
        )
        model = torch.load(save_path, weights_only=False)
        break

Validation Epoch: 1/100: 100%|██████████| 3/3 [00:00<00:00, 34.52it/s]


>> Epoch 1 Results:
   Train MSE: 1.28
   Val   MSE: 1.22 | RMSE: 0.72 Avg Mood Points




Validation Epoch: 2/100: 100%|██████████| 3/3 [00:00<00:00, 34.95it/s]


>> Epoch 2 Results:
   Train MSE: 1.09
   Val   MSE: 1.09 | RMSE: 0.73 Avg Mood Points




Validation Epoch: 3/100: 100%|██████████| 3/3 [00:00<00:00, 33.95it/s]


...

>> Epoch 4 Results:
   Train MSE: 1.02
   Val   MSE: 1.30 | RMSE: 0.79 Avg Mood Points




Validation Epoch: 5/100: 100%|██████████| 3/3 [00:00<00:00, 33.58it/s]


>> Epoch 5 Results:
   Train MSE: 0.95
   Val   MSE: 1.18 | RMSE: 0.81 Avg Mood Points




Validation Epoch: 6/100: 100%|██████████| 3/3 [00:00<00:00, 57.79it/s]

>> Epoch 6 Results:
   Train MSE: 0.91
   Val   MSE: 1.34 | RMSE: 0.86 Avg Mood Points



Early stopping triggered. No improvement for 3 epochs.


In [52]:
test_dataset = RnnDataset(test_df)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=1,
)

In [ ]:
model.eval()
with torch.no_grad():
    y_labels = []
    y_preds = []
    user_ids = []
    for x, y, u in tqdm(test_dataloader):
        y_preds.append(model(x).squeeze())
        y_labels.append(y.squeeze())
        user_ids.extend(u)

    y_labels = torch.hstack(y_labels).numpy()
    y_preds = torch.hstack(y_preds).numpy()

100%|██████████| 4/4 [00:00<00:00, 31.64it/s]


In [54]:
decoded_labels = [
    decode_label(preprocessor, user_idx, label)
    for label, user_idx in zip(y_labels, user_ids)
]
decoded_predictions = [
    decode_label(preprocessor, user_idx, prediction)
    for prediction, user_idx in zip(y_preds, user_ids)
]

print(
    f"Test RMSE: {np.mean((np.array(decoded_predictions) - np.array(decoded_labels)) ** 2) ** 0.5:.2f} Avg Mood Points"
)

Test RMSE: 0.65 Avg Mood Points


In [108]:
from plotly import graph_objects as go

In [109]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=decoded_labels,
        y=decoded_predictions,
        mode="markers",
        name="Predictions vs True",
    )
)

In [133]:
print(
    f"Test RMSE: {np.mean((np.array(y_labels) - np.array(y_preds)) ** 2) ** 0.5:.2f} Avg Mood Points"
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=y_labels, y=y_preds, mode="markers", name="Predictions vs True")
)

Test RMSE: 1.18 Avg Mood Points
